# Project #1
## Data Visualization (LECD) 2024/25

#### Student name: Luciana Rocha

## Instructions
- Do not modify the structure of this notebook!
- Fill-in the corresponding cells with the code or text depending on the objective
- You can use this template as a working notebook, or copy&paste your final code here
- It is recommended that you export and submit the rendered version of the notebook
- Submit the notebook not later than the date specified on the Inforestudante webpage of the course


## Data

MostPopularBaby_Names(NewYork)

The data includes the following features:

* brth_yr: birth year of the baby;
* gndr: baby's gender;
* ethcty: mother's ethnicity;
* nm: baby's name;
* cnt: count of the name;
* rnk: rank of the name.

The objective of this dataset and the study conducted throughout the visualization is to discover the most popular baby name for each ethnicity and gender.

For this purpose, the following variables were selected:

* baby's gender (gndr);
* ethnicity (ethcty);
* baby's name (nm);
* name count (cnt).

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import datetime as dt
import matplotlib.pyplot as plt
import plotly.graph_objects as go


file_path = 'most_popular_baby_names_by_sex_and_mother_s_ethnic_group_new_york_city.csv'
df = pd.read_csv(file_path)

## EDA Preliminar

In [2]:
print("Shape The DataSet ", df.shape )

Shape The DataSet  (13962, 6)


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13962 entries, 0 to 13961
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   brth_yr  13962 non-null  int64
 1   gndr     13962 non-null  str  
 2   ethcty   13962 non-null  str  
 3   nm       13962 non-null  str  
 4   cnt      13962 non-null  int64
 5   rnk      13962 non-null  int64
dtypes: int64(3), str(3)
memory usage: 654.6 KB


In [4]:
df.isna().sum()

brth_yr    0
gndr       0
ethcty     0
nm         0
cnt        0
rnk        0
dtype: int64

In [5]:
df.groupby('ethcty').size()

ethcty
ASIAN AND PACI                 358
ASIAN AND PACIFIC ISLANDER    1952
BLACK NON HISP                 353
BLACK NON HISPANIC            2274
HISPANIC                      4254
WHITE NON HISP                 701
WHITE NON HISPANIC            4070
dtype: int64

In [6]:
df.groupby('nm').size()

nm
AALIYAH    10
AARAV       5
AARON      20
AAYAN       1
ABBY        5
           ..
Zissy       2
Zoe         8
Zoey        8
Zoya        1
Zuri        2
Length: 2811, dtype: int64

## Data Cleaning

As can be observed, the ethnicities are duplicated but under different names: 

* ASIAN AND PACI & ASIAN AND PACIFIC ISLANDER
* BLACK NON HISP & BLACK NON HISPANIC
* WHITE NON HISP & WHITE NON HISPANIC

We will merge them as follows:

* ASIAN AND PACI with ASIAN AND PACIFIC ISLANDER
* BLACK NON HISP with BLACK NON HISPANIC
* WHITE NON HISP with WHITE NON HISPANIC

We can also see that the names have different casing, some in uppercase and others in lowercase:

ABBY≠Abby

So what we are going to do is convert everything to lowercase

In [7]:
df = df.replace(to_replace="ASIAN AND PACI", value='ASIAN AND PACIFIC ISLANDER')
df = df.replace(to_replace="BLACK NON HISP", value='BLACK NON HISPANIC')
df = df.replace(to_replace="WHITE NON HISP", value='WHITE NON HISPANIC')

In [8]:
df['nm'] = df['nm'].str.lower()
df['ethcty'] = df['ethcty'].str.lower()
df['gndr'] = df['gndr'].str.lower()

In [9]:
df = df.dropna(subset=['nm', 'ethcty', 'gndr', 'cnt'])

df_unique = df.drop_duplicates(subset=['nm', 'gndr', 'ethcty', 'cnt'])

## Data Transformation / Preprocessing

In [10]:
df['gndr'] = df['gndr'].astype('category')
df['ethcty'] = df['ethcty'].astype('category')

In [11]:
df_grouped = df.groupby(['nm', 'ethcty', 'gndr'], as_index=False).agg({'cnt': 'sum'})
print(df_grouped.head())

df_top_names = df_grouped.loc[df_grouped.groupby(['ethcty', 'gndr'])['cnt'].idxmax()]

        nm                      ethcty    gndr  cnt
0    aahil  asian and pacific islander    male   14
1  aaliyah          black non hispanic  female  471
2  aaliyah                    hispanic  female  431
3    aarav  asian and pacific islander    male   87
4    aaron  asian and pacific islander    male  399


## Visualization model #1

Scatter Plot 3D

In [12]:
fig_scatter_3d = go.Figure()

fig_scatter_3d.add_trace(go.Scatter3d(
    x=df_grouped['ethcty'],
    y=df_grouped['gndr'],
    z=df_grouped['cnt'],
    mode='markers',
    text=df_grouped['nm'],
    hoverinfo='text+name',
    marker=dict(
        size=10,
        color=df_grouped['cnt'],
        colorscale='Viridis',
        showscale=True
    )
  )
)

fig_scatter_3d.update_layout(
    title='Relação entre Etnia, Género e Número de Nomes',
    hovermode='closest',
    scene=dict(
        xaxis_title='Etnia',
        yaxis_title='Género',
        zaxis_title='Count'
    )
)

fig_scatter_3d.show()

## Visualization model #2

Grouped Bar Chart

In [13]:
fig = go.Figure()

for gndr in ['male', 'female']:
    df_gndr = df_top_names[df_top_names['gndr'] == gndr]
    fig.add_trace(go.Bar(
        x=df_gndr['ethcty'],
        y=df_gndr['cnt'],
        name=gndr.capitalize(),
        text=df_gndr['nm'],
        hoverinfo='text+name',
        textposition='auto'
    )
  )

fig.update_layout(
    title='Nome Mais Popular por Etnia e Género',
    xaxis_title='Etnia',
    yaxis_title='Count',
    barmode='group',
    xaxis_tickangle=-45,
    legend_title='Género'
)

fig.show()


## Visualization model #3

Dot Plot


In [14]:

fig_dot = go.Figure()

for gndr in df_top_names['gndr'].unique():
    df_gndr = df_top_names[df_top_names['gndr'] == gndr]
    fig_dot.add_trace(go.Scatter(
        x=df_gndr['ethcty'],
        y=df_gndr['cnt'],
        mode='markers+text',
        name=gndr.capitalize(),
        text=df_gndr['nm'],
        textposition='top center',
        marker=dict(
            size=30,
            color='blue' if gndr == 'male' else 'red',
            opacity=0.7
        )
    )
    )

fig_dot.update_layout(
    title='Nomes Mais Populares por Etnia e Género',
    xaxis_title='Etnia',
    yaxis_title='Contagem',
    legend_title='Género',
    hovermode='closest'
)

fig_dot.show()


## Visualization model #4

Bubble Chart

In [15]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_top_names['ethcty'],
    y=df_top_names['gndr'],
    mode='markers',
    text=df_top_names['nm'],
    marker=dict(
        size=df_top_names['cnt'] / 20,
        color=df_top_names['cnt'],
        colorscale='Viridis',
        showscale=True
    )
  )
)

fig.update_layout(
    title='Nome de Bebé Mais Usado por Etnia e Género',
    xaxis_title='Etnia',
    yaxis_title='Género',
    hovermode='closest'
)

fig.show()

## Critical Reflection & Evaluation

### 1. Analysis of Domain Findings
The aggregated results reveal distinct demographic patterns in baby naming conventions across New York City:
* **Gender Convergence vs. Cultural Differentiation:** 
  * Among **male babies**, there is a notable intercultural convergence: **"Jayden"** emerged as the single most popular name across three distinct ethnic groups (*Asian and Pacific Islander*, *Black Non-Hispanic*, and *Hispanic*), indicating widespread cross-cultural appeal and contemporary naming trends during this time window. Only *White Non-Hispanic* departed from this pattern with **"Joseph"**, reflecting enduring biblical and traditional naming conventions.
  * In contrast, **female baby names** showed complete ethnic divergence: each background presented a unique top name (**"Sophia"** for *Asian and Pacific Islander*, **"Madison"** for *Black Non-Hispanic*, **"Isabella"** for *Hispanic*, and **"Esther"** for *White Non-Hispanic*). This suggests that cultural specificity, linguistic heritage, and community traditions exert a stronger differentiating influence on female naming practices.
* **Volume Discrepancies:** The substantial difference in total counts across ethnicities (e.g., Hispanic and White Non-Hispanic populations showing markedly higher total counts than Asian and Black non-Hispanic groups) reflects NYC's overall demographic weight rather than genuine differences in naming enthusiasm, highlighting the need for proportional or normalized metrics in future extensions.

---

### 2. Comparative Evaluation of Visualization Models

Following established visual semiotics and human perceptual frameworks (Munzner, 2014; Cleveland & McGill, 1984; Tufte, 2001), the three tested models present key trade-offs:

| Criterion | Model #1: 3D Scatter Plot (`go.Scatter3d`) | Model #2: Grouped Bar Chart (`go.Bar`) | Model #3: Categorical Dot Plot (`go.Scatter`) |
| :--- | :--- | :--- | :--- |
| **Primary Visual Encoding** | 3D Cartesian coordinates + Viridis colormap | Length/height aligned to a common baseline | Position along an unaligned scale + dot size |
| **Perceptual Accuracy** | **Low:** Perspective foreshortening and occlusion hinder quantitative comparison | **High:** Aligned length is ranked among the highest perceptual channels (Cleveland & McGill, 1984) | **Medium-High:** Accurate point position, but large marker sizes introduce visual bounding noise |
| **Suitability for Goal** | Poor (visual overload, high cognitive burden for categorical data) | **Best (Optimal benchmark):** Direct side-by-side comparison of counts and name labels | Good (clean, uncluttered), but lacks intuitive volumetric anchoring compared to bars |

#### Model #1: 3D Scatter Plot
* **Shortcomings:** Projecting two categorical attributes (`ethcty`, `gndr`) onto discrete spatial planes in 3D creates massive occlusion and requires active user rotation to decipher data points. It incurs gratuitous cognitive overhead (the "3D trap" discussed by Munzner, 2014).
* **Verdict:** Unsuitable for answering the core analytical question, serving primarily as an exploratory check for global distribution.

#### Model #2: Grouped Bar Chart
* **Strengths:** By grouping male and female categories side-by-side on a shared zero-baseline, it leverages human perceptual acuity for length comparisons. Direct textual annotations (`textposition='auto'`) immediately identify the winning names without requiring tooltip inspection.
* **Verdict:** **Most effective model.** It satisfies Tufte's data-ink ratio principles and answers the research question instantly.

#### Model #3: Dot Plot
* **Strengths:** Significantly reduces ink/clutter compared to 3D and offers a sleek minimal view.
* **Shortcomings:** Using arbitrary marker sizes (size 30) without encoding an underlying quantitative dimension introduces perceptual distortion. Without baseline bars, fast magnitude estimation across separated columns is slightly harder than in Model #2.

---

### 3. Limitations and Future Improvements
1. **Normalization by Cohort Size:** Current absolute counts favor more populous ethnic groups. Computing the popularity index as a percentage of total births within each ethnic group would offer a fairer comparison.
2. **Temporal Dimension:** Collapsing all birth years (`brth_yr`) into an aggregate sum masks longitudinal shifts (e.g., names rising or fading across decades). A small-multiples or bump chart would effectively illustrate name dynamics over time.
3. **Data Integrity Note:** In the cleaning phase, deduplication and case normalization (`.str.lower()`) resolved duplicate records caused by legacy administrative labels (e.g., merging "ASIAN AND PACI" with "ASIAN AND PACIFIC ISLANDER").

## References

1. **Cleveland, W. S., & McGill, R. (1984).** Graphical Perception: Theory, Experimentation, and Application to the Development of Graphical Methods. *Journal of the American Statistical Association*, 79(387), 531–554. https://doi.org/10.1080/01621459.1984.10478080
2. **Munzner, T. (2014).** *Visualization Analysis and Design*. CRC Press, Taylor & Francis Group (A K Peters Visualization Series).
3. **Tufte, E. R. (2001).** *The Visual Display of Quantitative Information* (2nd ed.). Graphics Press.
4. **Few, S. (2012).** *Show Me the Numbers: Designing Tables and Graphs to Enlighten* (2nd ed.). Analytics Press.
5. **Plotly Technologies Inc. (2024).** *Plotly Open Source Graphing Library for Python*. https://plotly.com/python/
6. **NYC OpenData (2024).** *Popular Baby Names* [Data set]. Department of Health and Mental Hygiene (DOHMH), The City of New York. https://data.cityofnewyork.us/Health/Popular-Baby-Names/25th-nujf